In [1]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GaloisPoly
%aimport classes.BchEncoder

from classes.GaloisField import *
from classes.GaloisPoly import *
from classes.BchEncoder import *

## BCH (15, 5, t=3)
* n = 15 >>> if m = 4, (2**m - 1) % n = 0 >>> GF(16) (over GF2 symbols)
* g need 2*t = 6 consecutive roots >>> {a, a2, a4, a8, a3, a6, a12, a5, a10} = {C1, C3, C5}


In [3]:
m = 4
n = 15

# symbols and extended galois fields
gf_base     = GaloisField(1, 0b11)
gf_extended = GaloisField(m, 0b10011)

# print(gf_base)
# print(gf_extended)

# cyclotomic classes, generator poly
C1 = [1, 2, 4, 8]
C3 = [3, 6, 12, 9]
C5 = [5, 10]

aux = GaloisPoly(gf_extended, [ELEMENT_ONE])
p1  = aux.poly_from_roots(C1)
p3  = aux.poly_from_roots(C3)
p5  = aux.poly_from_roots(C5)
g   = p1*p3*p5

g


Field Closed Succesfully!, 1 Non-Zero Elements
Field Closed Succesfully!, 15 Non-Zero Elements


[0, -1, 0, -1, -1, 0, 0, -1, 0, 0, 0]

In [1]:
from classes.BchEncoder import *

# p(x) == x^10 + x^8 + x^5 + x^4 + x^2 + x + 1
gf_base = GaloisField(1, 0b11)
g       = GaloisPoly(gf_base, [0, -1, 0, -1, -1, 0, 0, -1, 0, 0, 0])
encoder = BchEncoder(15, g)

for w in range(2**encoder._k):
    u   = encoder.do_unpack(w, bit_width=encoder._k)
    v   = encoder.encode(u)
    print(v)
    cw  = encoder.get_output(w)
    assert(cw == code[w])

Field Closed Succesfully!, 1 Non-Zero Elements
Field Closed Succesfully!, 1 Non-Zero Elements
p(x) == [0]


NameError: name 'code' is not defined

In [5]:
code = [
    0b000000000000000   ,
    0b000010100110111   ,
    0b000101001101110   ,
    0b000111101011001   ,
    0b001000111101011   ,
    0b001010011011100   ,
    0b001101110000101   ,
    0b001111010110010   ,
    0b010001111010110   ,
    0b010011011100001   ,
    0b010100110111000   ,
    0b010110010001111   ,
    0b011001000111101   ,
    0b011011100001010   ,
    0b011100001010011   ,
    0b011110101100100   ,
    0b100001010011011   ,
    0b100011110101100   ,
    0b100100011110101   ,
    0b100110111000010   ,
    0b101001101110000   ,
    0b101011001000111   ,
    0b101100100011110   ,
    0b101110000101001   ,
    0b110000101001101   ,
    0b110010001111010   ,
    0b110101100100011   ,
    0b110111000010100   ,
    0b111000010100110   ,
    0b111010110010001   ,
    0b111101011001000   ,
    0b111111111111111   ]

# Correct systematic BCH(15,5,7) table

Convention:

codeword = [ m4 m3 m2 m1 m0 | p9 p8 p7 p6 p5 p4 p3 p2 p1 p0 ]

| Info    | Parity       | Codeword              |
| ------- | ------------ | --------------------- |
| 00000 | 0000000000 | **000000000000000** |
| 00001 | 0100110111 | **000010100110111** |
| 00010 | 1001101110 | **000101001101110** |
| 00011 | 1101011001 | **000111101011001** |
| 00100 | 0111101011 | **001000111101011** |
| 00101 | 0011011100 | **001010011011100** |
| 00110 | 1110000101 | **001101110000101** |
| 00111 | 1010110010 | **001111010110010** |
| 01000 | 1111010110 | **010001111010110** |
| 01001 | 1011100001 | **010011011100001** |
| 01010 | 0110111000 | **010100110111000** |
| 01011 | 0010001111 | **010110010001111** |
| 01100 | 1000111101 | **011001000111101** |
| 01101 | 1100001010 | **011011100001010** |
| 01110 | 0001010011 | **011100001010011** |
| 01111 | 0101100100 | **011110101100100** |
| 10000 | 1010011011 | **100001010011011** |
| 10001 | 1110101100 | **100011110101100** |
| 10010 | 0011110101 | **100100011110101** |
| 10011 | 0111000010 | **100110111000010** |
| 10100 | 1101110000 | **101001101110000** |
| 10101 | 1001000111 | **101011001000111** |
| 10110 | 0100011110 | **101100100011110** |
| 10111 | 0000101001 | **101110000101001** |
| 11000 | 0101001101 | **110000101001101** |
| 11001 | 0001111010 | **110010001111010** |
| 11010 | 1100100011 | **110101100100011** |
| 11011 | 1000010100 | **110111000010100** |
| 11100 | 0010100110 | **111000010100110** |
| 11101 | 0110010001 | **111010110010001** |
| 11110 | 1011001000 | **111101011001000** |
| 11111 | 1111111111 | **111111111111111** |


In [ ]:
import os
from pysv import compile_lib, generate_sv_binding
from classes.model import GoldModel

# # 0. Inject the compatibility policy directly into CMake's global configuration environment (Fedora 44, issue with cmake 4.3.0)
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
# os.environ["CXXFLAGS"] = "-Wno-deprecated"

# 1. Compile C++ lib from model
lib_path = compile_lib([GoldModel], cwd="build")
print(f"¡Success! C++ bridge generated in: {lib_path}")

# 2. Generate SystemVerilog package for UVM
generate_sv_binding([GoldModel], filename="pysv_pkg.sv")
print(f"¡Success! SystemVerilog package 'pysv_pkg.sv' ready for Vivado")

# BEFORE RUN VIVADO:
# 
#   Link vivado python3.13 interpreter to the python used for compilation (first time ever)
# 
#       cd < vivado_instalation >/tps/lnx64/
#       mv python-3.13.0 python-3.13.0.bak
#       ln -s /usr ./python-3.13.0
# 
#   Include python class directory
# 
#       cd < classes_directory >
#       export PYTHONPATH=$PYTHONPATH:$(pwd)
#       export LD_PRELOAD=/usr/lib64/libstdc++.so.6
# 
# INSIDE VIVADO:
#   
#   Include compiled lib for elaboration (first time ever)
# 
#       tools > settings > Simulation > Elaboration:
#       xelab.more_options: -sv_lib ../../../../../model/pysv/libpysv


CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Deprecation Warning at pybind11/CMakeLists.txt:8 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Warning (dev) at pybind11/tools/FindPythonLibsNew.cmake:98 (find_package):
  Policy CMP0148 is not set: The FindPythonInterp and FindPythonLibs modules
  are removed.  Run "cmake --help-policy CMP0148" for policy details.  Use
  the cmake_policy

-- pybind11 v2.11.1 
-- Configuring done (0.2s)
-- Generating done (0.0s)
-- Build files have been written to: /home/clifferto/Documents/carrerpath/projects/fec_work/arch/bch_15_5/encoder/model/build/build
[100%] Built target libpysv
¡Success! C++ bridge generated in: build/libpysv.so
¡Success! SystemVerilog package 'pysv_pkg.sv' ready for Vivado
